# Tăng cường ảnh thiếu sáng cho nhận diện biển báo

Bài tập lớn môn Xử lý ảnh - GTSRB, 43 lớp

Ảnh gốc → tạo ảnh tối → Gamma / HE / CLAHE / Retinex và ba enhancer pretrained → so sánh ảnh, histogram, PSNR/SSIM → kiểm chứng bằng ResNet
Train mới ba seed ResNet trên ảnh clean, rồi giữ cố định. Đánh giá mới dùng 430 ảnh test (10/lớp); đây là subset, không phải full-test 12.630 ảnh

## 1. Clone repo và cài môi trường
Chạy trên Kaggle, sử dụng 2 GPU T4.


In [ ]:
import os, sys, subprocess
from pathlib import Path
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
ROOT = Path('/tmp/gtsrb-notebook')
REPO = 'https://github.com/ZesyC/low-light-enhancement-for-traffic-sign-recognition.git'
subprocess.run(['git', 'clone', '--quiet', REPO, str(ROOT)], check=True)
subprocess.run(['git', '-C', str(ROOT), 'checkout', '--quiet', '82a7c2e4732c6bd0c0808999703e799a026f17c0'], check=True)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
BUNDLE = next(Path('/kaggle/input').rglob('notebook_bundle.json')).parent
sys.path.insert(0, str(BUNDLE))
import notebook_helpers as nb
config, device, output = nb.setup(ROOT, BUNDLE)


In [ ]:
from src.training import train
from src.experiment import tune, freeze, evaluate


## 2. Chuẩn bị dữ liệu
Dùng hàm chuẩn bị dữ liệu trong repo: chia train/validation theo track, crop ROI có padding 5% và resize RGB 64×64. Track chứa bản sao pixel của test được cách ly. Chọn ảnh minh họa từ validation và chốt subset test trước training.


In [ ]:
rows, test, examples, example, clean = nb.prepare_data(ROOT, config, BUNDLE)


## 3. Xử lý ảnh
Tạo ảnh tối: gamma > 1 làm giảm cường độ; Poisson mô phỏng nhiễu phụ thuộc tín hiệu; Gaussian bổ sung nhiễu cộng

Tăng sáng:
- Gamma: dùng số mũ < 1 để nâng vùng tối
- HE: cân bằng histogram độ sáng Y, giữ hai kênh màu
- CLAHE: cân bằng cục bộ; clipLimit hạn chế khuếch đại tương phản/nhiễu
- Multi-Scale Retinex: tách phản xạ và chiếu sáng ở nhiều thang Gaussian


In [ ]:
dark = nb.show_degradation(clean, example, config)


In [ ]:
nb.show_classical(clean, dark['medium'], config)


In [ ]:
nb.show_parameters(dark['medium'])


## 4. Train ResNet để kiểm chứng
- train mới ba seed 11/22/33 trên ảnh clean
- ResNet-18 khởi tạo ImageNet, AdamW, batch 32, tối đa 20 epoch và early stopping theo validation và không train ảnh tối/enhanced

In [ ]:
checkpoints = [train(ROOT, config, seed, device, output/'train') for seed in [11,22,33]]
nb.show_training(output)

## 5. Chọn tham số trên validation
Dùng 1.290 ảnh validation để chọn Gamma, CLAHE, Retinex và pipeline; không chọn bằng test. Zero-DCE, Retinexformer và Diffusion dùng weights tác giả để tăng sáng, không train lại enhancer trên GTSRB.


In [ ]:
selection = tune(ROOT, config, checkpoints, device, output/'validation', per_class=30)
nb.show_validation(selection)

In [ ]:
nb.show_enhancers(clean, dark['medium'], example, selection, device, config)

## 6. Đánh giá và đọc kết quả
- PSNR/SSIM: mức giữ lại thông tin ảnh. 
- Macro-F1: kiểm chứng khả năng nhận diện đủ 43 lớp. So với identity để thấy tăng sáng giúp hay làm hỏng nhận diện. Mean ± SD được tính qua ba training seed sau khi trung bình noise trong mỗi seed.

In [ ]:
protocol = freeze(ROOT, output/'validation/selection.json', output/'protocol_subset.json', per_class=10)
evaluate(ROOT, output/'protocol_subset.json', output/'test_subset', device)

In [ ]:
nb.show_results(ROOT, output)

## 7. Ablation, demo và xuất kết quả
Tách gamma-only / gamma + Gaussian / gamma + Poisson / gamma + cả hai để giải thích tác động của từng bước xử lý ảnh. Giữ nguyên classifier và tham số đã chọn: 288 lượt trên cùng 430 ảnh. Khi tắt Poisson, chuỗi RNG Gaussian cũng thay đổi; gamma-only không tạo ba ảnh độc lập.

Demo dùng một ảnh validation cố định mỗi lớp. Các cell phụ dưới đây chỉ gọi lại hàm xử lý ảnh và dự đoán từ repo. Kết luận giới hạn ở thiếu sáng mô phỏng sRGB, không suy ra hiệu quả camera đêm thực tế.

In [ ]:
nb.run_ablation(ROOT, config, selection, checkpoints, device, test)

In [ ]:
nb.run_demo(ROOT, config, selection, checkpoints, device, examples)

In [ ]:
nb.export_results(ROOT, output, selection, checkpoints)